# Cribl Python SDK\n\nUse `cribl-control-plane` to fetch inventory and turn it into quick operational visuals.

## What you'll learn\n\n1. Install and initialize the SDK\n2. Fetch health plus source/destination/pipeline counts\n3. Build a baseline matplotlib chart\n4. Use AI prompts with Jinja summaries for iteration

### Setup\n\nRun the next cell **once** before the rest. The SDK needs **httpx** and **pydantic**; those packages ship as **WASM wheels inside Pyodide**, not as manylinux wheels on PyPI. The cell pins them to the same versions Pyodide bundles (see [Pyodide built-in packages](https://pyodide.org/en/stable/usage/packages-in-pyodide.html)) so `micropip` never tries to download incompatible Rust wheels or bad proxy pages. Then it installs `cribl-control-plane` (pure Python). Matplotlib ships with the kernel.\n\nWhen you upgrade the app\u2019s Pyodide release, refresh those pins to match the table.

In [ ]:
import micropip\n\n# Match Pyodide 0.29.x built-in stack (WASM). Update when bumping PYODIDE_RELEASE.\nawait micropip.install([\n    'httpcore==1.0.7',\n    'httpx==0.28.1',\n    'pydantic-core==2.41.5',\n    'pydantic==2.12.5',\n    'cribl-control-plane',\n])

In [ ]:
import os\nserver_url = os.getenv('CRIBL_API_URL')\nif not server_url:\n    raise RuntimeError('CRIBL_API_URL is not set in this runtime.')\nserver_url

In [ ]:
from cribl_control_plane import CriblControlPlane\n\nhealth = {}\napi_summary = {}\n\nwith CriblControlPlane(server_url=server_url) as ccp:\n    group_url = f'{server_url}/m/default'\n    health = ccp.health.get()\n    sources = ccp.sources.list(server_url=group_url)\n    destinations = ccp.destinations.list(server_url=group_url)\n    pipelines = ccp.pipelines.list(server_url=group_url)\n\napi_summary = {\n    'group': 'default',\n    'source_count': len((sources.items if sources else None) or []),\n    'destination_count': len((destinations.items if destinations else None) or []),\n    'pipeline_count': len((pipelines.items if pipelines else None) or []),\n}\napi_summary

## Checkpoint\n\nYou should see non-error values in `api_summary` before plotting.

In [ ]:
import matplotlib.pyplot as plt\n\nlabels = ['sources', 'destinations', 'pipelines']\nvalues = [api_summary['source_count'], api_summary['destination_count'], api_summary['pipeline_count']]\n\nplt.figure(figsize=(7, 4))\nplt.bar(labels, values)\nplt.title(f"Cribl object counts ({api_summary['group']})")\nplt.ylabel('count')\nplt.tight_layout()\nplt.show()

## Next steps\n\n- Open `AI_Magic.ipynb` to practice iterative prompt refinement.\n- Open `Incident_Triage_Playbook.ipynb` for a broader triage workflow.

In [ ]:
# ### Prompt:\n# Generate improved visualization code for API inventory trends.\n# Context:\n# - health: object:\n#          {{ health | describe }}\n# - api_summary: object\n#          {{ api_summary | describe }}\n#\n# Requirements:\n# - Use the existing objects, don't create samples\n# - choose chart type automatically\n# - include title/labels\n# - print 2-3 observations